# 01. ROS — 규칙 기반 손 추종 / 60분

**선행:** 07.Mediapipe 손 관절, 08.ROS topic·turtlesim

**이 노트북의 결과:** 손 관절 규칙과 손 위치를 구분하고 `target_xy()`를 구현합니다. 규칙 기반 추종을 실행해 이후 VLM 비교의 기준을 남깁니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

손 모양은 이동 허용 여부를, 손 위치는 추종할 목표를 정합니다. 손을 왼쪽으로 옮기는 것은 LEFT 명령을 보내는 분류 과제가 아니라 화면 좌표를 거북이 공간의 목표로 바꾸는 일입니다. 현재 거북이 방향에 따라 회전 후 전진하고 가까워지면 멈춥니다.

### 전체 과정에서 내 코드가 하는 일

```text
MediaPipe 손 좌표
  ↓
비주먹/주먹 규칙
  ↓
화면 목표 TODO
  ↓
거북이 pose와 방향·거리
  ↓
속도
  ↓
overlay·로그
```

정규화 좌표의 범위·Y 반전·mirror를 구현합니다. 실제 ROS 실행기는 제공된 공통 제어기를 사용하므로 학생 함수 검증과 live 관찰 결과를 각각 기록합니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 빨간 점이 있는데 거북이가 안 움직이면 인식 실패인가요?

**설명:** 반드시 그렇지는 않습니다. 좌표는 있지만 목표 도달·주먹·판단 만료 등으로 정지할 수 있습니다. 점 색만 보지 말고 action의 reason과 pose를 확인합니다. 초록은 회전도 포함한 이동이며 좌표가 없을 때는 NO TARGET입니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**4일차 환경:** `nepes-course:physical-ai-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 카메라·GUI는 로컬 화면에서 함께 관찰하고 입력 프로세스는 하나만 실행합니다.

**VLM 실습 준비:** VLM 질의에는 공용 Ollama의 `http://127.0.0.1:11434` API와 `gemma4:e2b-it-qat` 모델이 필요합니다. 해당 실습의 연결 확인에서 실패하면 로컬 Setup 담당자에게 서버·모델 준비를 요청합니다. 순수 Python fixture와 YOLO 실습에는 Ollama가 필요하지 않습니다.

**작업 순서:** 아래 첫 설정 셀의 `ROOT`가 `/workspace`인지 확인 → 현재 노트북이 지정한 `exercises/*.py`를 Jupyter 편집기에서 저장 → `student = load_file(...)` 검증 셀을 다시 실행합니다. 새 커널에는 이전 노트북의 변수나 import가 없으므로 이 노트북 자체의 설정·fixture 셀을 사용합니다. 실행 중 보이는 출력은 학습 확인용이고 배포본에는 저장하지 않습니다. TODO 실패와 카메라·모델 환경 실패를 구분하세요.

## 슬라이드와 함께 진행하기

시작 안내는 [4일차 슬라이드 10쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s005) · [배포 PDF 10쪽](../슬라이드/4일차.pdf#page=10)입니다. 노트북의 설명·TODO·검증·실제 실행을 진행한 뒤, [4일차 슬라이드 11쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s047) · [배포 PDF 11쪽](../슬라이드/4일차.pdf#page=11)로 돌아가 11–18쪽에서 결과를 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.


## 정답 활용 안내

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이번 실습이 전체 흐름에서 맡는 역할

[01 규칙 이동](01_ros_gesture_action.ipynb) → [02 VLM 프롬프트 검증](02_vlm_decision.ipynb) → [03 VLM 손 추종](03_ros_vlm_action.ipynb) → [04 판단 유지시간](04_ros_vlm_latency.ipynb) → [05 YOLO detection·pose](05_yolo_detection_pose.ipynb) → [06 pose 좌표·LLM·ROS](06_ros_pose_llm.ipynb). [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.

**구현 실습과 실제 실행의 관계:** 지정한 `exercises/` 파일에 학생이 알고리즘을 구현하고 노트북 검증 셀로 평가합니다. `solutions/`는 별도 참고 정답이며, 참고 정답 검증만 실행해도 학생 TODO가 완성되는 것은 아닙니다. 카메라·ROS 데모는 제공된 실행용 구현을 사용합니다. 학생 파일 저장이 실행 중인 ROS 노드를 자동으로 바꾸지는 않습니다. 실시간 동작과 자신의 함수 검증을 각각 완료하세요.


## 시간 지도

관절·주먹 10분 → 좌표 TODO·검증 15분 → ROS 준비 10분 → 규칙 추종 20분 → 기록 5분


## 필요한 앞 내용 — 위치와 핵심을 함께 확인

[07.Mediapipe/02_hand_gestures.ipynb의 손 관절 실습](../07.Mediapipe/02_hand_gestures.ipynb)에 정규화 좌표가 있습니다. 손 관절 좌표는 x·y 각각 0~1이며 이 실습의 손 중심은 관절 0/5/9/13/17 평균입니다. raw 영상은 x가 오른쪽, y가 아래로 커집니다. 한 손의 유효한 중심만 목표로 사용하며 두 손·미검출은 중심이 없습니다.

시간 보호 기본값은 촬영 후 5000ms, 승인 후 400ms, pose 갱신 후 500ms 미만입니다. 자세한 유지 시간 비교는 04에서 진행합니다. 여기의 fixture와 공통 제어기는 자체 import하므로 이전 학생 TODO를 먼저 완성하지 않아도 좌표 TODO를 검증할 수 있습니다. 실제 손 추종에는 `/workspace/Data/models/07.Mediapipe/hand_landmarker.task`가 필요합니다.


In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "09.Pseudo-VLA").is_dir():
            return candidate
    raise RuntimeError("09.Pseudo-VLA를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "09.Pseudo-VLA"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('cv2', 'ultralytics', 'rclpy', 'requests')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 손 모양은 이동 방향이 아니다

<!-- procedure:lesson-30-dff05c:start -->
![규칙으로 손 위치를 추종하기](../교재그림/절차/lesson-30.svg)

손 모양은 이동 방향이 아닙니다. 좌표가 목표를 정하고 상태·유효성 검사가 이동 허용을 정합니다.
<!-- procedure:lesson-30-dff05c:end -->


비주먹의 모든 손 모양은 같은 FOLLOW 의미입니다. 좌우/전후는 손바닥 **위치** 가 결정합니다.
손목→손가락 끝 거리와 손목→중간 관절 거리의 비율로 펴짐을 추정하며 엄지도 검사합니다.
한 손만 검출된 경우를 사용합니다. 두 손, 작은 손, 좌표 범위 밖, 판정 경계는 STOP입니다.
이는 기하 규칙이라 손 가림·원근·회전에 실패할 수 있습니다. handedness 점수를 제스처 확률로 사용하지 않습니다.

원본 영상 x는 오른쪽, y는 아래입니다. 거북이 x는 오른쪽, y는 위입니다.
`target_x = 1 + 9*x`, `target_y = 1 + 9*(1-y)`로 벽에서 여유를 둡니다.
거울 모드이면 x를 `1-x`로 **한 번만** 바꿉니다.
거북이는 목표로 순간이동하지 않고 pose의 현재 위치·방향으로 이동합니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.hand_follow import rule_decision
observation = dict(frame_id="f1", captured_ms=1000, center=[1.0, 0.5], state="NON_FIST")
decision = rule_decision(observation)
pose = (5.5, 5.5, 0.0)
print(observation, decision, pose)
from common.hand_follow import follow_motion
print(follow_motion(observation, decision, pose, now_ms=1100, issued_ms=1050, pose_ms=1100))


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## TODO — [exercises/hand_target.py](exercises/hand_target.py)

`target_xy(center, mirror=False)`를 구현합니다. center는 0–1의 유한한 숫자 두 개여야 합니다.
bool, NaN, 범위 밖을 ValueError로 거부합니다. 출력은 (x,y) tuple입니다.
손을 화면 각 모서리로 옮겼을 때 목표를 먼저 예측하세요.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** `target_xy(center,mirror=False)`는 `[x,y]` 또는 `(x,y)` 정규화 좌표를 받아 turtlesim 내부 1–10 범위의 `(tx,ty)`를 반환합니다. `[0,0]`→`(1,10)`, `[1,1]`→`(10,1)`, `[.5,.5]`→`(5.5,5.5)`입니다.
**작성 흐름 힌트:** 길이2·bool 제외·유한수·0–1 검사 → mirror면 x를 1-x로 변경 → x를 1+9x로 확대/이동 → y를 1+9(1-y)로 반전. y 반전은 한 번만 합니다.
**설명할 지점:** “화면은 아래로 갈수록 y가 커지고 turtlesim은 위로 갈수록 커집니다. 방향 변환과 범위 변환을 나누어 생각하세요.”
<!-- todo-review:requirements:end -->

### TODO의 입력·출력·경계 전체 계약

작성 위치는 `/workspace/09.Pseudo-VLA/exercises/hand_target.py`의 `target_xy(center, *, mirror=False)`입니다. center는 list 또는 tuple, 길이는 정확히 2입니다. x·y는 bool이 아닌 int/float 유한수이고 **0과 1을 포함** 합니다. 길이 오류·문자열·NaN·Inf·범위 밖 값은 ValueError로 거부합니다. 잘못된 값을 중앙이나 경계로 고쳐 반환하지 않습니다. mirror는 수업에서 True/False로 전달합니다.

출력은 숫자 두 개의 tuple `(tx,ty)`입니다. 입력을 수정하지 않고 x 미러링(선택) → `[0,1]`을 `[1,10]`으로 확대·이동 → y 방향 반전을 수행합니다. 별도의 ROS 호출·속도·주먹 판정을 작성하지 않습니다.

| center | mirror=False | mirror=True |
| --- | --- | --- |
| (0,0) | (1,10) | (10,10) |
| (1,1) | (10,1) | (1,1) |
| (0.5,0.5) | (5.5,5.5) | (5.5,5.5) |

**성공 기준:** 이 세 점과 mirror 결과, 비정상 입력 거부, 입력 불변이 학생 검증 셀에서 통과해야 합니다. 출력은 거북이의 목표 위치이며 즉시 이동하거나 `cmd_vel`을 발행하지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_target", "exercises/hand_target.py")
assert student.target_xy([0, 0]) == (1, 10)
assert student.target_xy([1, 1]) == (10, 1)
assert student.target_xy([0, 0], mirror=True) == (10, 10)
try: student.target_xy([True, 0])
except ValueError: pass
else: raise AssertionError("bool must be rejected")
center = [0.5, 0.5]
assert student.target_xy(center) == (5.5, 5.5)
assert student.target_xy(center, mirror=True) == (5.5, 5.5)
assert student.target_xy([1, 1], mirror=True) == (1, 1)
assert center == [0.5, 0.5]
for invalid in ([], [0], [0, 0, 0], "00", ["0", 0], [float("nan"), 0],
                [float("inf"), 0], [-0.1, 0], [0, 1.1]):
    try:
        student.target_xy(invalid)
    except ValueError:
        pass
    else:
        raise AssertionError(f"좌표를 거부해야 합니다: {invalid!r}")
print("PASS target mapping")


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 참고 정답

강사가 안내하는 시점에 아래 접힌 정답 셀을 여세요. 셀은 `solutions/`의 독립적인 참고 구현이 같은 계약을 통과하는지 확인합니다. 바로 아래 정답 해설·접힌 코드에 입력 검사·경계 처리·반환값을 다시 제공합니다. 실제 카메라·ROS 공통 구현은 별도 runtime이며 참고 정답 실행과 학생 TODO 검증은 각각 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_target", "solutions/hand_target.py")
assert reference.target_xy([0, 0]) == (1, 10)
assert reference.target_xy([.5, .5]) == (5.5, 5.5)
print("PASS reference")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/hand_target.py](solutions/hand_target.py)의 `target_xy`입니다. center가 길이 2이고 x·y가 bool이 아닌 유한한 0–1 숫자인지 확인합니다. x는 `1+9*x`, y는 `1+9*(1-y)`로 1–10 내부 좌표에 대응합니다.

(0,0)→(1,10), (0.5,0.5)→(5.5,5.5), (1,1)→(10,1)입니다. 영상 y는 아래로 증가하고 turtlesim y는 위로 증가하므로 한 번 반전합니다. mirror=True이면 x도 1-x로 한 번 반전합니다.

범위를 벗어난 좌표를 중앙값으로 숨기지 않고 오류로 알립니다. 손 위치 목표와 실제 거북이 속도 계산은 다른 함수 책임입니다.

**확인 질문:** 입력 영상을 이미 좌우 반전했는데 목표에서도 또 반전하면 사용자가 보는 움직임이 어떻게 달라지나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/hand_target.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: 정규화한 손 중심 (x,y)을 turtlesim 내부 좌표로 변환합니다.
x 미러링과 아래로 증가하는 영상 y의 반전을 별도로 처리합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
import math

def number(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
        raise ValueError('finite number required')
    return float(value)


def target_xy(center, *, mirror=False):
    """Raw camera x right / y down → turtlesim interior [1,10]², y up."""
    if not isinstance(center, (list, tuple)) or len(center) != 2:
        raise ValueError('center must have x,y')
    x, y = map(number, center)
    if not 0 <= x <= 1 or not 0 <= y <= 1:
        raise ValueError('center outside image')
    # x의 선택적 mirror와 y의 위아래 반전을 분리하고 내부 범위 1~10으로 옮깁니다.
    return (1 + 9*(1-x if mirror else x), 1 + 9*(1-y))
```

</details>
<!-- todo-review:answer:end -->


## 실제 ROS 실행 — JupyterLab 터미널 A·B·C

<!-- procedure:ros-terminals-57bc75:start -->
![새 터미널 준비와 세 역할](../교재그림/절차/ros-terminals.svg)

1–3을 새 터미널마다 적용합니다. 4–6은 동시에 실행하는 역할이며 노드·토픽 이름은 해당 실습 명령을 따릅니다.
<!-- procedure:ros-terminals-57bc75:end -->


새 Terminal에서는 아래 블록의 첫 줄 bash부터 전체를 복사합니다. A·B·C 모두 같은 배정 Domain(예:21)을 사용합니다. 노트북 Python 셀에 붙이지 않습니다. 이미 환경을 설정한 같은 B에서 모드·TTL만 바꿀 때는 Ctrl+C 후 launch 명령만 실행합니다.

**처음 준비 또는 패키지를 찾지 못할 때 — 빌드:**
```bash
bash
bash /workspace/08.ROS/scripts/build_workspace.sh
bash /workspace/09.Pseudo-VLA/scripts/build_workspace.sh
```

**A — 카메라 입력:** 이미 A가 실행 중이면 그대로 유지합니다. 카메라 publisher는 하나만 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 run v4l2_camera v4l2_camera_node --ros-args -p video_device:=/dev/video0 -p image_size:="[640, 480]" -r image_raw:=/camera/image_raw
```

카메라 번호는 배정값입니다. 녹화 입력으로 비교할 때는 A를 Ctrl+C로 종료하고 같은 A에서 아래 명령을 실행합니다. 매 조건에서 같은 시작 지점으로 재생을 다시 시작합니다.
```bash
ros2 run nepes_ros_lab recorded_image_publisher --ros-args -p image_source:=/workspace/Data/videos/07.Mediapipe/person_open_palm.avi
```

**거북이의 움직임:** 한 손이 확실한 비주먹이면 손 중심을 목표로 삼습니다. 손을 영상 오른쪽/위로 옮기면 거북이 목표도 오른쪽/위로 이동합니다. 주먹·손 없음·복수 손·불확실 판단이면 정지합니다. 목표와 방향 차이가 0.6 rad 이상이면 제자리 회전 후 전진하며, 목표 거리 0.15 이하에서 멈춥니다. 손가락이 가리키는 방향을 읽거나 일정 시간 무조건 전진하는 동작이 아닙니다.

**화면 읽기:** /vla/image_overlay의 초록색 점은 전진·회전 중, 빨간색 점은 좌표가 있지만 정지 중입니다. 좌표가 없으면 NO TARGET입니다. Target은 거북이 목표 좌표, age는 제어에 사용한 프레임 경과 시간입니다. 승인 경과 시간·TTL·남은 유효시간도 확인합니다. VLM 추론 중에는 이전 관측 화면이 유지됩니다. 최신 원본은 /camera/image_raw입니다. mirror:=true는 목표 x만 반전하고 영상과 점은 원본 위치를 유지합니다.

**B — 규칙 기반 이동:** VLM은 실행하지 않습니다. 한 손·주먹·비주먹으로 기준 동작을 관찰합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 launch nepes_vla_demo hand_follow.launch.py mode:=rule \
  ttl_ms:=400 \
  log_path:=/workspace/Data/logs/09.Pseudo-VLA/01-hand-rule.jsonl
```

**C — 관측·판단·행동 확인:** 새 Terminal에서 전체를 실행합니다.
```bash
bash
source /opt/ros/humble/setup.bash
source /workspace/Data/build/08.ROS/install/setup.bash
source /workspace/Data/build/09.Pseudo-VLA/install/setup.bash
export ROS_DOMAIN_ID=21
ros2 topic echo /vla/action
```

연속 출력은 Ctrl+C로 끝낸 뒤, 같은 C에서 필요에 따라 ros2 topic echo /vla/perception 또는 ros2 topic echo /vla/decision을 실행합니다. frame_id·reason·TTL·승인 경과 시간을 영상과 비교합니다.

**종료:** A 중단 → C에서 STOP 확인 → B 종료 → C 종료. 다음 노트북으로 바로 이어갈 때는 A를 유지하고 B만 종료합니다. GUI 없는 자습은 B launch 앞에 QT_QPA_PLATFORM=offscreen, 인자에 with_image_view:=false를 사용합니다.

학생 TODO는 Python 검증 셀로 평가합니다. 실제 ROS는 제공 공통 구현을 실행하며 학생 파일 변경이 자동 반영되지는 않습니다.


## 완료 검사

1. 손을 좌·중앙·우·위·아래로 옮기고 Target과 거북이 이동을 비교합니다.
2. 큰 방향 차이면 먼저 회전하고 목표 거리 0.15 이하에서 멈추는지 확인합니다.
3. 손을 고정한 채 주먹·비주먹을 바꾸고 점 색과 정지 이유를 기록합니다.
4. mirror:=true로 B만 재실행하여 좌우 목표를 비교합니다. 03 VLM 비교 전에는 기본 false로 복귀합니다.
5. 가림·원근에 따른 규칙 실패와 01-hand-rule.jsonl을 기록합니다.

다음 [02 VLM 프롬프트 검증](02_vlm_decision.ipynb)에서는 카메라 제어를 잠시 멈추고 이미지에 대한 JSON 응답이 제어 입력 계약을 만족하는지 확인합니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

[4일차 슬라이드 11쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s047) · [배포 PDF 11쪽](../슬라이드/4일차.pdf#page=11)에서 11–18쪽을 복습합니다. 자신의 TODO 검증과 실제 관측·미측정을 나누어 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습
손 관절의 흔들림을 기록하고 이동 평균을 시험할 수 있습니다. 필터가 추가한 반응 지연과 정지 반응도 함께 비교합니다.
